# 02 — Reglas de asociación (Apriori)

Transacciones clínicas → medicamento. Los umbrales de soporte y confianza se eligen en una validación interna dentro de Entrenamiento (Hit@3, cobertura, MRR, Hit@1); después las reglas se reconstruyen con todo Entrenamiento y se evalúan una sola vez en Prueba. Incluye la cobertura, la longitud de las listas y el desempeño en casos cubiertos (observación 8).

In [ ]:
!pip -q install mlxtend scikit-learn pandas numpy

In [ ]:
import io, os, json, random, warnings
import numpy as np, pandas as pd
from google.colab import files
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, KFold, RandomizedSearchCV, train_test_split
warnings.filterwarnings("ignore")
RS = 42
random.seed(RS); np.random.seed(RS)
SPLIT = 'fila'   # 'fila' = análisis principal del artículo | 'grupo' = sensibilidad agrupada por episodio
FEATURES = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']
TARGET = 'MEDICAMENTOS'
KS = (1, 3, 5)
LOG = {}
print("Sube Tacotalpa2024.csv")
up = files.upload(); CSV = next(iter(up))
OUT = f"salida_{SPLIT}"
os.makedirs(OUT, exist_ok=True)

## Preparación común (idéntica en los cuatro notebooks)
Limpieza, episodio aproximado, descripción de clases y partición 80/20 (semilla 42). Con `SPLIT='fila'` se reproduce la partición del artículo (6,277 / 1,560); con `SPLIT='grupo'`, la partición agrupada por episodio usada como análisis de sensibilidad.

In [ ]:
# ---------------------------------------------------------------- 1. Flujo de datos
df = pd.read_csv(CSV, encoding="latin1")
flow = [("Registros en el archivo integrado (3 hospitales, 2020-2022)", len(df))]
for c in ['AFECPRIN', 'AFEC', 'MEDICAMENTOS', 'DESCRIPCION', 'SEXO', 'TIPOURGENCIA']:
    df[c] = df[c].astype("string").str.strip()
df['EDAD'] = pd.to_numeric(df['EDAD'], errors='coerce')
n0 = len(df)
df = df.dropna(subset=FEATURES + [TARGET]).copy()
flow.append(("Excluidos por valores faltantes en variables esenciales", -(n0 - len(df))))
# Episodio aproximado: filas consecutivas con la misma EDAD, SEXO, TIPOURGENCIA y AFECPRIN (el archivo
# integrado conserva el orden de la unión URGENCIAS-AFECCIONES-MEDICAMENTOS, pero no el ID original).
# AFEC no forma parte de la clave porque la unión genera el producto AFEC x MEDICAMENTO dentro de un episodio.
EP_KEY = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN']
key = df[EP_KEY].astype(str).agg('|'.join, axis=1)
df['EPISODIO'] = (key != key.shift()).cumsum().astype(int)
ndup = int(df.duplicated(subset=FEATURES + [TARGET, 'DESCRIPCION']).sum())
df = df.drop_duplicates(subset=FEATURES + [TARGET, 'DESCRIPCION']).reset_index(drop=True)
flow.append(("Excluidos por duplicados exactos", -ndup))
# Un medicamento repetido dentro de un mismo episodio (distinta presentación) se cuenta una vez
# para el nivel episodio; a nivel fila se conserva como en el protocolo original.
flow.append(("Conjunto analítico (filas episodio-medicamento)", len(df)))
LOG['n_analitico'] = len(df)
LOG['n_episodios'] = int(df.EPISODIO.nunique())
LOG['n_perfiles_unicos'] = int(df[FEATURES].astype(str).agg('|'.join, axis=1).nunique())
ep_size = df.groupby('EPISODIO')[TARGET].nunique()
LOG['episodios_con_1_med'] = int((ep_size == 1).sum())
LOG['episodios_con_2plus_med'] = int((ep_size >= 2).sum())
LOG['filas_en_episodios_multi'] = int(df.EPISODIO.map(df.groupby('EPISODIO').size()).gt(1).sum())
LOG['episodios_multi_afec'] = int((df.groupby('EPISODIO').AFEC.nunique() > 1).sum())
LOG['max_meds_episodio'] = int(ep_size.max())

# ---------------------------------------------------------------- 2. Descripción de clases
vc = df[TARGET].value_counts()
LOG['n_clases'] = int(len(vc))
LOG['clases_1_obs'] = int((vc == 1).sum())
LOG['clases_lt5'] = int((vc < 5).sum())
LOG['clases_lt10'] = int((vc < 10).sum())
for k in (1, 3, 5, 10, 20):
    LOG[f'prop_top{k}'] = float(vc.head(k).sum() / len(df))
p = vc / vc.sum()
LOG['entropia_bits'] = float(-(p * np.log2(p)).sum())
LOG['perplejidad'] = float(2 ** LOG['entropia_bits'])
vc.rename_axis('Medicamento').reset_index(name='Frecuencia').assign(
    Porcentaje=lambda d: 100 * d.Frecuencia / len(df),
    Acumulado=lambda d: d.Porcentaje.cumsum()).to_csv(f"{OUT}/distribucion_medicamentos.csv", index=False)

# ---------------------------------------------------------------- 3. Partición
singleton = vc[vc < 2].index
ep_single = set(df.loc[df[TARGET].isin(singleton), 'EPISODIO'])
if SPLIT == 'grupo':
    pool = df[~df.EPISODIO.isin(ep_single)]
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS)
    tr_idx, te_idx = next(sgkf.split(pool, pool[TARGET], groups=pool.EPISODIO))
    test_df = pool.iloc[te_idx].copy()
    train_df = pd.concat([pool.iloc[tr_idx], df[df.EPISODIO.isin(ep_single)]]).copy()
else:  # protocolo original: estratificado por fila, sin agrupar
    df_single = df[df[TARGET].isin(singleton)]
    df_split = df[~df[TARGET].isin(singleton)]
    tr, test_df = train_test_split(df_split, test_size=.2, random_state=RS, stratify=df_split[TARGET])
    train_df = pd.concat([tr, df_single]).copy()
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
LOG['split'] = SPLIT
LOG['n_train'] = len(train_df); LOG['n_test'] = len(test_df)
LOG['ep_train'] = int(train_df.EPISODIO.nunique()); LOG['ep_test'] = int(test_df.EPISODIO.nunique())
LOG['ep_compartidos'] = len(set(train_df.EPISODIO) & set(test_df.EPISODIO))
LOG['filas_singleton_train'] = int(df[TARGET].isin(singleton).sum())
LOG['filas_en_ep_singleton_train'] = int(df.EPISODIO.isin(ep_single).sum())
LOG['clases_test_no_vistas'] = int((~test_df[TARGET].isin(set(train_df[TARGET]))).sum())
flow.append(("Entrenamiento (filas)", len(train_df)))
flow.append(("Prueba interna reservada (filas)", len(test_df)))
pd.DataFrame(flow, columns=['Etapa', 'Registros']).to_csv(f"{OUT}/flujo_datos.csv", index=False)

# Representación a nivel episodio para la prueba: un perfil y el conjunto de medicamentos.
test_ep = test_df.groupby('EPISODIO').agg(**{c: (c, 'first') for c in FEATURES},
                                           MEDS=(TARGET, lambda s: set(s))).reset_index()
LOG['test_episodios'] = len(test_ep)

In [ ]:
# ---------------------------------------------------------------- utilidades de métricas
def row_hits(y, recs):
    """Matriz por caso: rango (1..5) del medicamento verdadero o 0 si no aparece."""
    return np.array([(r.index(t) + 1) if t in r else 0 for t, r in zip(y, recs)])


def set_ranks(sets, recs):
    """Nivel episodio: rango del primer medicamento relevante del episodio."""
    out = []
    for s, r in zip(sets, recs):
        pos = [i + 1 for i, m in enumerate(r) if m in s]
        out.append(pos[0] if pos else 0)
    return np.array(out)


def summarize(ranks, recs, sets=None):
    d = {f'Hit@{k}': float(np.mean((ranks > 0) & (ranks <= k))) for k in KS}
    d['MRR'] = float(np.mean(np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0)))
    d['Cobertura'] = float(np.mean([len(r) > 0 for r in recs]))
    if sets is not None:  # recall@5 a nivel episodio
        d['Recall@5'] = float(np.mean([len(set(r[:5]) & s) / len(s) for s, r in zip(sets, recs)]))
    return d

In [ ]:
RECS = {}  # (recomendaciones por fila, recomendaciones por episodio)
test_ep_ids = test_ep.EPISODIO

## Transacciones, reglas y recomendación

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

def age_band(a):
    for lo, hi, lab in [(0, 11, '00_11'), (12, 17, '12_17'), (18, 29, '18_29'), (30, 44, '30_44'),
                        (45, 59, '45_59'), (60, 74, '60_74'), (75, 200, '75_MAS')]:
        if lo <= a <= hi:
            return lab
    return 'DESCONOCIDA'


def tok(p, v):
    return f"{p}={str(v).strip().replace('|', '/')}"


def transaction(r, med=True):
    x = [tok('EDADGRUPO', age_band(float(r['EDAD']))), tok('SEXO', r['SEXO']), tok('URGENCIA', r['TIPOURGENCIA']),
         tok('AFECPRIN', r['AFECPRIN']), tok('AFEC', r['AFEC'])]
    if med:
        x.append(tok('MED', r[TARGET]))
    return x


def make_rules(data, s, c, max_len=4):
    tx = [transaction(r) for _, r in data.iterrows()]
    te = TransactionEncoder(); basket = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
    freq = apriori(basket, min_support=s, use_colnames=True, max_len=max_len, low_memory=True)
    if freq.empty:
        return pd.DataFrame(), 0
    try:
        r = association_rules(freq, metric='confidence', min_threshold=c)
    except Exception:
        return pd.DataFrame(), len(freq)
    if r.empty:
        return r, len(freq)
    ok = r.apply(lambda z: len(z.consequents) == 1 and next(iter(z.consequents)).startswith('MED=')
                 and all(not q.startswith('MED=') for q in z.antecedents) and z.lift > 1, axis=1)
    r = r[ok].sort_values(['confidence', 'lift', 'support'], ascending=False).reset_index(drop=True)
    return r, len(freq)


def ap_recommend(row, rules, k=5):
    if rules is None or rules.empty:
        return []
    patient = set(transaction(row, False)); best = {}
    for z in rules.itertuples():
        if set(z.antecedents) <= patient:
            med = next(iter(z.consequents)).replace('MED=', '', 1)
            score = (z.confidence, z.lift, z.support, len(z.antecedents))
            if med not in best or score > best[med]:
                best[med] = score
    # orden: confianza, lift, soporte, longitud del antecedente; desempate final alfabético
    return [m for m, _ in sorted(best.items(), key=lambda x: (tuple(-v for v in x[1]), x[0]))[:k]]

## Selección de soporte y confianza (solo Entrenamiento)

In [ ]:
# Validación interna dentro de Train (agrupada por episodio)
if SPLIT == 'grupo':
    vc_tr = train_df[TARGET].value_counts()
    ep_s = set(train_df.loc[train_df[TARGET].isin(vc_tr[vc_tr < 2].index), 'EPISODIO'])
    pool = train_df[~train_df.EPISODIO.isin(ep_s)]
    a, b = next(StratifiedGroupKFold(5, shuffle=True, random_state=RS).split(pool, pool[TARGET], pool.EPISODIO))
    subtrain = pd.concat([pool.iloc[a], train_df[train_df.EPISODIO.isin(ep_s)]]); val = pool.iloc[b]
else:
    vc_tr = train_df[TARGET].value_counts(); spl = vc_tr[vc_tr >= 2].index
    tsp = train_df[train_df[TARGET].isin(spl)]
    sm, val = train_test_split(tsp, test_size=.2, random_state=RS, stratify=tsp[TARGET])
    subtrain = pd.concat([sm, train_df[~train_df[TARGET].isin(spl)]])
grid = []
for s in [.001, .0025, .005, .01]:
    for c in [.30, .40, .50, .60, .70]:
        rl, ni = make_rules(subtrain, s, c)
        rc = [ap_recommend(r, rl) for _, r in val.iterrows()]
        grid.append({'min_support': s, 'min_confidence': c, 'itemsets': ni, 'reglas': len(rl),
                     **summarize(row_hits(val[TARGET].tolist(), rc), rc)})
grid = pd.DataFrame(grid).sort_values(['Hit@3', 'Cobertura', 'MRR', 'Hit@1', 'min_confidence'],
                                      ascending=False).reset_index(drop=True)
grid.to_csv(f"{OUT}/apriori_busqueda.csv", index=False)
S, C = float(grid.iloc[0].min_support), float(grid.iloc[0].min_confidence)
display(grid.head(10))

## Reglas finales y evaluación en Prueba

In [ ]:
rules, nit = make_rules(train_df, S, C)
LOG['ap_support'] = S; LOG['ap_conf'] = C; LOG['ap_itemsets'] = nit; LOG['ap_rules'] = len(rules)
rx = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rx['antecedents'] = rx.antecedents.apply(lambda x: ' & '.join(sorted(x)))
rx['consequents'] = rx.consequents.apply(lambda x: ' & '.join(sorted(x)))
rx.to_csv(f"{OUT}/apriori_reglas.csv", index=False)
RECS['Apriori'] = ([ap_recommend(r, rules) for _, r in test_df.iterrows()],
                   [ap_recommend(r, rules) for _, r in test_ep.iterrows()])
print('Soporte:', S, '| Confianza:', C, '| Itemsets:', nit, '| Reglas:', len(rules))
display(rx.head(20))

In [ ]:
rk = row_hits(test_df[TARGET].tolist(), RECS['Apriori'][0])
display(pd.DataFrame([summarize(rk, RECS['Apriori'][0])]).round(4))
recs = RECS['Apriori'][0]; cov = np.array([len(r) > 0 for r in recs])
print('Cobertura:', round(100*cov.mean(), 2), '%')
print('En casos cubiertos:', summarize(rk[cov], [r for r in recs if r]))
ln = pd.Series([len(r) for r in recs]).value_counts().sort_index()
display(pd.DataFrame({'longitud_lista': ln.index, 'casos': ln.values, 'porcentaje': (100*ln.values/ln.sum()).round(2)}))

## Exportación

In [ ]:
pred = pd.DataFrame({'EPISODIO': test_df.EPISODIO, 'Real': test_df[TARGET],
                     'Top5': [' | '.join(r) for r in RECS['Apriori'][0]]})
pred.to_csv(f"{OUT}/apriori_predicciones_test.csv", index=False)
with open(f"{OUT}/apriori_resumen.json", "w", encoding="utf-8") as f:
    json.dump(LOG, f, ensure_ascii=False, indent=2, default=float)
print(json.dumps(LOG, ensure_ascii=False, indent=1, default=float))
files.download(f"{OUT}/apriori_predicciones_test.csv"); files.download(f"{OUT}/apriori_resumen.json")
files.download(f"{OUT}/apriori_reglas.csv"); files.download(f"{OUT}/apriori_busqueda.csv")